# 풀스택 GPT: #5.0~5.8

## Tasks:

- [x] 앞서 배운 메모리 클래스 중 하나를 사용하는 메모리로 LCEL 체인을 구현합니다.
    > - [x] [ConversationBufferMemory](https://python.langchain.com/v0.1/docs/modules/memory/types/buffer/) 등 강의에서 배운 메모리 중 하나를 사용하여 이전 대화 기록을 기억하고 기록을 이용한 답변을 제공할 수 있도록 합니다.
    > - [x] 채팅 형식의 메모리 기록을 프롬프트에 추가하고 싶을 때는 [MessagesPlaceholder](https://python.langchain.com/v0.1/docs/modules/memory/adding_memory/#adding-memory-to-a-chat-model-based-llmchain)를 이용하세요.
    > - [x] RunnablePassthrough를 활용하면 LCEL 체인을 구현할 때 메모리 적용을 쉽게 할 수 있습니다. RunnablePassthrough는 메모리를 포함한 데이터를 체인의 각 단계에 전달하는 역할을 합니다. (강의 #5.7 1:04~ 참고)
- [x] 이 체인은 영화 제목을 가져와 영화를 나타내는 세 개의 이모티콘으로 응답해야 합니다. (예: "탑건" -> "🛩️👨‍✈️🔥". "대부" -> "👨‍👨‍👦🔫🍝").
- [x] 항상 세 개의 이모티콘으로 답장하도록 [FewShotPromptTemplate](https://python.langchain.com/v0.1/docs/modules/model_io/prompts/few_shot_examples/) 또는 [FewShotChatMessagePromptTemplate](https://python.langchain.com/v0.1/docs/modules/model_io/prompts/few_shot_examples_chat/)을 사용하여 체인에 예시를 제공하세요.
- [x] 메모리가 작동하는지 확인하려면 체인에 두 개의 영화에 대해 질문한 다음 다른 셀에서 체인에 먼저 질문한 영화가 무엇인지 알려달라고 요청하세요.

In [3]:
examples = [
    {
        "question":"Top Gun",
        "answer":"🛩️👨‍✈️🔥",
    },
    {
        "question":"The Godfather",
        "answer":"👨‍👨‍👦🔫🍝"
    },
]

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain.chat_models.openai import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate, MessagesPlaceholder
from langchain.schema.runnable import RunnablePassthrough

memory = ConversationBufferMemory(
    return_messages=True,
)

llm = ChatOpenAI(
    model="gpt-4o-mini-2024-07-18",
    temperature=0.2,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}"),
    ("ai", "{answer}"),
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system","Human give title of a movie and You reply with three emojis that represent the movie."),
    few_shot_prompt,
    MessagesPlaceholder(variable_name="history"),
    ("human","{question}"),
])

def load_memory(_):
    return memory.load_memory_variables({})["history"]

chain = RunnablePassthrough.assign(history=load_memory) | final_prompt | llm

def invoke_chain(title):
    result = chain.invoke({"question": title})
    memory.save_context({"input":title}, {"output":result.content})

invoke_chain("Exhuma")
invoke_chain("Midsommar")

🪦🔍👻🌼🌞🔪

In [10]:
chain.invoke({"question":"What was the name of the movie I've asked right before?"})

The movie you asked about right before was "Midsommar."

[HumanMessage(content='Exhuma'),
 AIMessage(content='🪦🔍👻'),
 HumanMessage(content='Midsommar'),
 AIMessage(content='🌼🌞🔪')]

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain.chat_models.openai import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts import PromptTemplate, FewShotPromptTemplate
from langchain.schema.runnable import RunnablePassthrough

memory = ConversationBufferMemory(
    memory_key="history",
)

llm = ChatOpenAI(
    temperature=0.2,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

example_template = """
    Human: {question}
    AI: {answer}
"""

example_prompt = PromptTemplate.from_template(example_template)

few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Human: {question}",
    input_variables=["question"],
)

template = """
    Human give title of a movie and You reply with three emojis that represent the movie.

    {few_shot_prompt}
    {history}
    Human: {question}
    You: 
"""

final_prompt = PromptTemplate.from_template(template)

def load_memory(_):
    history = memory.load_memory_variables({})["history"]
    return history

chain = {
    "few_shot_prompt": few_shot_prompt,
    "history": load_memory,
    "question": RunnablePassthrough()      # pass the raw question along
} | final_prompt | llm

def invoke_chain(title):
    result = chain.invoke({"question": title})
    memory.save_context({"input":title},{"output":result.content})

invoke_chain("Avatar")
invoke_chain("My Octopus Teacher")

🌿🔵🪐🐙👨‍🏫🌊

In [13]:
chain.invoke({"question":"What was the first movie i've asked?"})

Top Gun: 🛩️👨‍✈️🔥

AIMessageChunk(content='Top Gun: 🛩️👨\u200d✈️🔥')